# 🔄 Scheduled Retraining Pipeline — DVC + MLflow

**Project 2 of 10 — MLOps Portfolio Series**  
**Author:** Jumma Mohammad Teli | Birmingham, UK  
**Dataset:** UCI Bank Marketing — 3 real quarterly batches (41,188 rows total)  
**Goal:** Automatically retrain on new data, compare to champion, promote only if AUC improves

---

## Notebook Structure
1. Setup & Imports
2. Data Versioning — Real Quarterly Splits
3. EDA — Drift Analysis Across Versions
4. Retraining Pipeline Execution
5. Champion/Challenger Results
6. Audit Trail Analysis
7. Key Takeaways

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

print('✅ All imports successful')
print(f'MLflow version: {mlflow.__version__}')

## 2. Data Versioning — Real Quarterly Splits

In [ ]:
# Generate versioned datasets
from src.data.generator import generate_all_versions

manifests = generate_all_versions(base_dir='../data')
manifest_df = pd.DataFrame(manifests)
print('\n=== DATA VERSION MANIFEST ===')
print(manifest_df.to_string(index=False))

In [ ]:
# Load all versions
dfs = {}
for v in [1, 2, 3]:
    path = f'../data/v{v}/bank_marketing_v{v}.csv'
    if os.path.exists(path):
        dfs[v] = pd.read_csv(path)
        print(f'v{v}: {dfs[v].shape[0]:,} rows | positive rate: {(dfs[v]["y"]=="yes").mean():.1%} | MD5: {manifests[v-1]["md5"][:8]}...')

## 3. EDA — Drift Analysis Across Versions

In [ ]:
# Target distribution across versions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#2563EB', '#16A34A', '#DC2626']
quarters = ['Q1 (Jan-Mar)', 'Q2 (Apr-Jun)', 'Q3 (Jul-Sep)']

for i, (v, ax) in enumerate(zip([1,2,3], axes)):
    if v in dfs:
        counts = (dfs[v]['y'] == 'yes').value_counts()
        pos_rate = (dfs[v]['y'] == 'yes').mean()
        ax.bar(['No', 'Yes'], [1-pos_rate, pos_rate], color=['#94A3B8', colors[i]], edgecolor='white')
        ax.set_title(f'v{v} — {quarters[i]}\n({dfs[v].shape[0]:,} rows)', fontweight='bold')
        ax.set_ylabel('Rate')
        ax.set_ylim(0, 1)
        ax.text(1, pos_rate + 0.02, f'{pos_rate:.1%}', ha='center', fontweight='bold')

plt.suptitle('Target Distribution Drift Across Quarters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Key: v1 has 50.5% positive rate — Jan-Mar targeted campaign period')

In [ ]:
# Contact method drift
if all(v in dfs for v in [1,2,3]):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, (v, ax) in enumerate(zip([1,2,3], axes)):
        contact_dist = dfs[v]['contact'].value_counts(normalize=True)
        contact_dist.plot(kind='bar', ax=ax, color=colors[i], edgecolor='white')
        ax.set_title(f'v{v} Contact Method\n{quarters[i]}', fontweight='bold')
        ax.set_ylabel('Proportion')
        ax.tick_params(rotation=0)
    plt.suptitle('Contact Method Distribution Drift', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Retraining Pipeline Execution

In [ ]:
# Run full retraining pipeline
import types
from src.retrain import run_retraining

mlflow.set_tracking_uri('../mlruns')
results = []

for version in [1, 2, 3]:
    print(f'\nRetraining on v{version}...')
    args = types.SimpleNamespace(
        data_version=version,
        random_state=42,
        min_improvement=0.005
    )
    metrics, promoted = run_retraining(args)
    results.append({'version': version, 'auc': metrics['roc_auc'], 'f1': metrics['f1'], 'promoted': promoted})

print('\n✅ Pipeline complete')

## 5. Champion/Challenger Results

In [ ]:
# Results table
results_df = pd.DataFrame(results)
results_df['quarter'] = ['Q1 Jan-Mar', 'Q2 Apr-Jun', 'Q3 Jul-Sep']
results_df['promoted_str'] = results_df['promoted'].map({True: '✅ YES', False: '❌ NO'})
print('=== CHAMPION/CHALLENGER RESULTS ===')
print(results_df[['version','quarter','auc','f1','promoted_str']].to_string(index=False))

In [ ]:
# AUC progression chart
fig, ax = plt.subplots(figsize=(10, 5))
versions = [f'v{r["version"]}\n{["Q1","Q2","Q3"][i]}' for i, r in enumerate(results)]
aucs = [r['auc'] for r in results]
bars = ax.bar(versions, aucs, color=['#2563EB','#16A34A','#DC2626'], edgecolor='white', width=0.5)
ax.axhline(y=0.8174, color='orange', linestyle='--', linewidth=2, label='P1 Baseline (0.8174)')
ax.set_ylabel('AUC Score')
ax.set_title('Champion AUC Progression Across Data Versions', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend()
for bar, auc in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{auc:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Audit Trail Analysis

In [ ]:
# Read audit trail
audit_path = '../reports/audit_trail.jsonl'
if os.path.exists(audit_path):
    audit_records = []
    with open(audit_path) as f:
        for line in f:
            audit_records.append(json.loads(line))
    audit_df = pd.DataFrame(audit_records)
    print('=== AUDIT TRAIL ===')
    print(audit_df[['timestamp','data_version','roc_auc','improvement','promoted','n_train']].tail(6).to_string(index=False))
else:
    print('Run the pipeline first to generate audit trail')

## 7. Key Takeaways

In [ ]:
print('=' * 60)
print('KEY TAKEAWAYS — Scheduled Retraining Pipeline')
print('=' * 60)
print('''
DATASET
  Real UCI Bank Marketing split into 3 quarterly batches
  v1 (Jan-Mar): 546 rows  | positive rate: 50.5%
  v2 (Apr-Jun): 21,719 rows | positive rate: 9.1%
  v3 (Jul-Sep): 13,922 rows | positive rate: 11.2%

RESULTS
  v1 AUC: 0.6397 → Promoted (first run)
  v2 AUC: 0.7719 → Promoted (+0.1322 improvement)
  v3 AUC: 0.7828 → Promoted (+0.0109 improvement)

MLOPS CONCEPTS
  ✅ Data versioning   — DVC + MD5 hashing
  ✅ Champion gate     — only promote if AUC improves
  ✅ Audit trail       — complete data→model lineage
  ✅ Scheduled runs    — weekly GitHub Actions cron
  ✅ Real data drift   — quarterly distribution shifts

NEXT STEPS
  Project 3: Feature Engineering as Versioned Artifact
  Project 7: Drift Detection Dashboard (Evidently AI)
''')
print('=' * 60)

---
**Author:** Jumma Mohammad Teli | Data Analyst & ML Engineer | Birmingham, UK  
**LinkedIn:** https://linkedin.com/in/jumma-mohammad  
**GitHub:** https://github.com/jumma786  

*Project 2 of 10 — MLOps Portfolio Series*